In [41]:
import pandas as pd
from pathlib import Path
import os
import numpy as np

# Merge macsiq annotations in a single file

In [42]:
#INPUTS
datadir=Path('D:/phenotyping project/pilot_10/N327_ROI1_L2')
signature_matrix=Path("D:/phenotyping project/pilot_08/macsiq_classification/parsed_data/pheno_table_ROI_01.csv")
#OUT
concatenated_file_path= datadir/ 'concatenated_files'/ 'macsiq_phenotyping.csv'


In [43]:


def concat_macsiq(workdir,output_file):
    files = list( workdir.glob("*.csv") )

    cell_ids=[ list(pd.read_csv(file)[['Cell Id']].values.flatten()) for file in files]
    
    phenotypes=[len(val)*[file.stem.split(' ',1)[-1]] for file,val in zip(files,cell_ids) ]      
    
    data=[ pd.DataFrame({'Cell Id':ids, 'cell_type':types }) for ids,types in zip(cell_ids,phenotypes)]
    
    all_data=pd.concat(data,ignore_index=True)
    int_labels=[ int(label.split('@')[0]) for label in list(all_data['Cell Id'].values)]
    all_data['Cell Id']=int_labels
    all_data.sort_values(by=['Cell Id'],inplace=True,ignore_index=True)
    df_macsiq=all_data.loc[:,['Cell Id','cell_type']]
    
    output_file.parent.mkdir(parents=True, exist_ok=True)
    df_macsiq.to_csv(output_file,index=False)
    
    return df_macsiq
    


In [44]:
if os.path.exists(concatenated_file_path):
    df_macsiq=pd.read_csv(concatenated_file_path)
else:
    df_macsiq=concat_macsiq(datadir,concatenated_file_path)

# Attach lineage info from celesta-like phenotype table

In [45]:
lineage_info={}
level=[]
source=[]
cell_type_label=[]
full_signat_matrix=pd.read_csv(signature_matrix)

for cell_type,lineage in zip(full_signat_matrix.iloc[:,0].values,full_signat_matrix.iloc[:,1].values):
    
    lineage_info[cell_type]=tuple(np.array(lineage.split('_')).astype('int'))
    
for c in df_macsiq.cell_type.values:
    info=lineage_info[c]
    level.append(info[0])
    source.append(info[1])
    cell_type_label.append(info[2])
    
df_lineage=pd.DataFrame({'level':level,'parent_cell':source,'type_no': cell_type_label })
df_macsiq=pd.concat([df_macsiq,df_lineage],ignore_index=False,axis=1)
df_macsiq.head()

,Cell Id,cell_type,level,parent_cell,type_no
0,1,Ungated,1,0,15
1,2,T cells,1,0,1
2,3,Cancer cells,1,0,11
3,4,Cancer cells,1,0,11
4,5,Cancer cells,1,0,11


In [46]:
output_dir=datadir/ 'concatenated_files'
filename='final_phenotyping_labels.csv'
pheno_depth=int(3)
df_macsiq_filt=df_macsiq.loc[df_macsiq['level']<=pheno_depth]


repeated_labels=[cellID for cellID, rep in (df_macsiq_filt['Cell Id'].value_counts()>1).items() if rep==True]
repeated_labels.sort()

remove_indices=[]
for cellID in repeated_labels:
    cell_lineage=df_macsiq_filt.loc[df_macsiq_filt['Cell Id']==cellID].level
    index=np.argmax(cell_lineage.values)
    final_type_index=cell_lineage.index.values[index]
    remove_elements=np.setdiff1d(cell_lineage.index.values,[final_type_index]).tolist()
    if remove_elements:
        remove_indices.extend(remove_elements)

df_macsiq_filt=df_macsiq_filt.drop(index=remove_indices)
df_macsiq_filt.to_csv(output_dir / filename ,index=False)
df_macsiq_filt.head()

,Cell Id,cell_type,level,parent_cell,type_no
0,1,Ungated,1,0,15
1,2,T cells,1,0,1
2,3,Cancer cells,1,0,11
3,4,Cancer cells,1,0,11
4,5,Cancer cells,1,0,11


In [54]:
#marker_path=Path("D:/phenotyping project/pilot_08/macsiq_classification/parsed_data/markers_mean_intensity_ROI_02_deep.csv")
#pheno_path=Path("D:/phenotyping project/pilot_08/macsiq_classification/ROI_02_deep/concatenated_files/final_phenotyping_labels.csv")
marker_path=Path("D:/phenotyping project/pilot_10/parsed_data/markers_mean_intensity_N327_ROI1_L2.csv")
pheno_path=Path("D:/phenotyping project/pilot_10/N327_ROI1_L2/concatenated_files/final_phenotyping_labels.csv")

data=pd.read_csv(marker_path)
pheno=pd.read_csv(pheno_path)

if len(data['Cell Id'])==len(pheno['Cell Id']):
    check_diff=data['Cell Id'].values-pheno['Cell Id'].values
    
    if not np.any(check_diff):
        data.insert(1,'cell_type',pheno['cell_type'].values)
        data.to_csv(pheno_path.parent / 'feature_table.csv',index=False)
    else:
        print('labels in marker intensity and cell type tables dont match')
    
else:
    print('labels in marker intensity and cell type tables dont match')

    
    

In [55]:
a=

SyntaxError: invalid syntax (62411553.py, line 1)